# 02 · Energy Analysis ⭐

This is the core analysis notebook for the thesis. It provides a deep dive into CPU and
Memory energy consumption across 18 languages, grouped by execution paradigm (AOT, JIT,
Interpreted).

**Units:** All energy values are in **Joules (J)** (converted from raw µJ at load time).

**Key questions:**
- Which languages are most energy-efficient?
- Do paradigms (AOT vs JIT vs Interpreted) differ significantly in energy consumption?
- How does energy vary across benchmarks?

**Methodology:**
- Rankings, heatmaps and summary tables use the **two-step mean** (mean per
  language × benchmark, then averaged across the 8 benchmarks with equal weight),
  sourced from `results_clean_runs.csv` via `lang_means()`.
- Significance testing keeps **non-parametric** tests, which are robust to the
  right-skewed, non-normal benchmark distributions:
  - **Kruskal-Wallis** (non-parametric ANOVA) for paradigm comparisons
  - **Mann-Whitney U** (pairwise) with **Bonferroni correction** for post-hoc tests
  - **Rank-biserial correlation** as the effect size measure
- Boxplots/violins are shown as distribution views (the box still shows the
  median/quartiles); they are ordered by the mean and the mean is marked (▲).

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))   # make the shared style module importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
from itertools import combinations

import plot_style as ps
ps.apply_style()

# Canonical constants — single source: plot_style.
COL_CPU_ENERGY, COL_MEM_ENERGY = ps.COL_CPU_ENERGY, ps.COL_MEM_ENERGY
COL_TIME                       = ps.COL_TIME
COL_CPU_CARBON, COL_MEM_CARBON = ps.COL_CPU_CARBON, ps.COL_MEM_CARBON
PARADIGM        = ps.PARADIGM
PARADIGM_COLORS = ps.PARADIGM_COLORS
PARADIGM_ORDER  = ps.PARADIGM_ORDER
MEANPROPS       = ps.MEANPROPS
ALPHA           = ps.ALPHA

OUTPUTS_DIR = Path('outputs'); OUTPUTS_DIR.mkdir(exist_ok=True)

# Single source of truth: per-run rows (df) + per-cell means with EDP (df_mean).
df      = ps.load_runs()
df_mean = ps.cell_means(df)

def lang_means(cols):
    """Per-language two-step mean (equal benchmark weight) for column(s) `cols`."""
    return ps.lang_means(df_mean, cols)

print(f"Runs: {df.shape} | Cell-means: {df_mean.shape} | "
      f"{df['language'].nunique()} languages \u00d7 {df['benchmark'].nunique()} benchmarks")
df_mean.head(3)

## 1. CPU Energy by Language

Boxplots sorted by **mean** CPU energy (J), with the mean marked (▲). Lower is better
(more energy-efficient). The box still shows the median/quartiles as a distribution
reference.

In [ ]:
# Order and annotate by the two-step mean (equal benchmark weight); the boxplot
# itself is drawn from the per-run df to show the distribution.
cpu_mean = lang_means(COL_CPU_ENERGY)
lang_order_cpu = cpu_mean.sort_values().index.tolist()

fig, ax = plt.subplots(figsize=(15, 6))
bp = ax.boxplot(
    [df[df['language'] == lang][COL_CPU_ENERGY].values for lang in lang_order_cpu],
    labels=lang_order_cpu, patch_artist=True, notch=False, showmeans=True, meanprops=MEANPROPS,
    medianprops=dict(color='black', linewidth=2),
    flierprops=dict(marker='x', markerfacecolor='red', markersize=5, alpha=0.6),
)
for patch, lang in zip(bp['boxes'], lang_order_cpu):
    patch.set_facecolor(PARADIGM_COLORS[PARADIGM[lang]])
    patch.set_alpha(0.75)

for i, lang in enumerate(lang_order_cpu):
    m = cpu_mean[lang]
    ax.text(i + 1, m, f'{m:.1f}', ha='center', va='bottom', fontsize=7, color='black')

legend_handles = [mpatches.Patch(color=PARADIGM_COLORS[p], label=p, alpha=0.75)
                  for p in PARADIGM_ORDER]
ax.legend(handles=legend_handles, title='Paradigm', loc='upper left')
ax.set_title('CPU Energy by Language (sorted by mean; ▲ = mean)', fontsize=13)
ax.set_xlabel('Language')
ax.set_ylabel('CPU Energy (J)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
ps.save_fig(fig, '02_cpu_energy_by_language')
plt.show()

> **Takeaway:** AOT-compiled languages (C++, C, Rust) cluster at the bottom (most efficient); interpreted languages (Perl, Python) sit at the top with the widest spread.

### CPU Energy per Paradigm Group

The same CPU-energy distribution as above, split by paradigm to show within-group spread (mean ▲).

In [ ]:
# Per-paradigm split of the distribution (same per-run data as the all-language
# boxplot above, grouped by paradigm to show within-group spread; mean = ▲).
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=False)
for ax, paradigm in zip(axes, PARADIGM_ORDER):
    langs = [l for l in lang_order_cpu if PARADIGM[l] == paradigm]
    data  = [df[df['language'] == l][COL_CPU_ENERGY].values for l in langs]
    bp = ax.boxplot(data, labels=langs, patch_artist=True, showmeans=True, meanprops=MEANPROPS,
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='x', markerfacecolor='red', markersize=5, alpha=0.6))
    for patch in bp['boxes']:
        patch.set_facecolor(PARADIGM_COLORS[paradigm])
        patch.set_alpha(0.75)
    pass
    ax.set_title(paradigm)
    ax.set_ylabel('CPU Energy (J)' if paradigm == 'AOT' else '')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
fig.suptitle('CPU Energy per Paradigm Group (J)', fontsize=13)
plt.tight_layout()
ps.save_fig(fig, '02_cpu_energy_per_paradigm')
plt.show()

## 2. Memory Energy by Language

Same structure as CPU energy. Memory energy (J) reflects DRAM power draw, which varies
less across paradigms but reveals GC pressure and allocation patterns.

In [ ]:
mem_mean = lang_means(COL_MEM_ENERGY)
lang_order_mem = mem_mean.sort_values().index.tolist()

fig, ax = plt.subplots(figsize=(15, 6))
bp = ax.boxplot(
    [df[df['language'] == lang][COL_MEM_ENERGY].values for lang in lang_order_mem],
    labels=lang_order_mem, patch_artist=True, showmeans=True, meanprops=MEANPROPS,
    medianprops=dict(color='black', linewidth=2),
    flierprops=dict(marker='x', markerfacecolor='red', markersize=5, alpha=0.6),
)
for patch, lang in zip(bp['boxes'], lang_order_mem):
    patch.set_facecolor(PARADIGM_COLORS[PARADIGM[lang]])
    patch.set_alpha(0.75)

for i, lang in enumerate(lang_order_mem):
    m = mem_mean[lang]
    ax.text(i + 1, m, f'{m:.3f}', ha='center', va='bottom', fontsize=7)

legend_handles = [mpatches.Patch(color=PARADIGM_COLORS[p], label=p, alpha=0.75)
                  for p in PARADIGM_ORDER]
ax.legend(handles=legend_handles, title='Paradigm', loc='upper left')
ax.set_title('Memory Energy by Language (sorted by mean; ▲ = mean)', fontsize=13)
ax.set_xlabel('Language')
ax.set_ylabel('Memory Energy (J)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
ps.save_fig(fig, '02_mem_energy_by_language')
plt.show()

> **Takeaway:** memory (DRAM) energy is an order of magnitude smaller than CPU energy and separates the paradigms far less — Erlang stands out, driven by its very long regex-redux run.

### Memory Energy per Paradigm Group

The same memory-energy distribution as above, split by paradigm to show within-group spread (mean ▲).

In [ ]:
# Per-paradigm split of the distribution (same per-run data as the all-language
# boxplot above, grouped by paradigm to show within-group spread; mean = ▲).
fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=False)
for ax, paradigm in zip(axes, PARADIGM_ORDER):
    langs = [l for l in lang_order_mem if PARADIGM[l] == paradigm]
    data  = [df[df['language'] == l][COL_MEM_ENERGY].values for l in langs]
    bp = ax.boxplot(data, labels=langs, patch_artist=True, showmeans=True, meanprops=MEANPROPS,
                    medianprops=dict(color='black', linewidth=2),
                    flierprops=dict(marker='x', markerfacecolor='red', markersize=5, alpha=0.6))
    for patch in bp['boxes']:
        patch.set_facecolor(PARADIGM_COLORS[paradigm])
        patch.set_alpha(0.75)
    pass
    ax.set_title(paradigm)
    ax.set_ylabel('Memory Energy (J)' if paradigm == 'AOT' else '')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
fig.suptitle('Memory Energy per Paradigm Group (J)', fontsize=13)
plt.tight_layout()
ps.save_fig(fig, '02_mem_energy_per_paradigm')
plt.show()

## 3. CPU + Memory Energy Combined

Two complementary views:
1. **Stacked bar chart** — total energy (CPU + Memory) per language in Joules, split by component
2. **Scatter plot** — CPU vs Memory energy (J), to identify languages where one dominates

In [ ]:
agg = lang_means([COL_CPU_ENERGY, COL_MEM_ENERGY])
agg = agg.sort_values(COL_CPU_ENERGY)
agg['paradigm'] = agg.index.map(PARADIGM)

fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(agg))
ax.bar(x, agg[COL_CPU_ENERGY], label='CPU Energy (J)', color='#666666', alpha=0.9)
ax.bar(x, agg[COL_MEM_ENERGY], bottom=agg[COL_CPU_ENERGY],
       label='Memory Energy (J)', color='#D55E00', alpha=0.9)

# total (CPU + memory) labelled on top of each stacked bar
totals = agg[COL_CPU_ENERGY] + agg[COL_MEM_ENERGY]
for xi, t in zip(x, totals):
    ax.text(xi, t + totals.max() * 0.01, f'{t:,.0f}',
            ha='center', va='bottom', fontsize=7, color='#333333')
ax.set_ylim(0, totals.max() * 1.08)

ax.set_xticks(x)
ax.set_xticklabels(agg.index, rotation=45, ha='right')
ax.set_title('Total Energy (CPU + Memory) by Language — mean across all benchmarks', fontsize=12)
ax.set_ylabel('Energy (J)')
ax.legend()
plt.tight_layout()
ps.save_fig(fig, '02_total_energy_stacked')
plt.show()

> **Takeaway:** CPU energy dominates the total; memory energy is a thin slice on top, so the energy ranking tracks the CPU ranking closely.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for paradigm in PARADIGM_ORDER:
    langs = [l for l in agg.index if agg.loc[l, 'paradigm'] == paradigm]
    ax.scatter(
        agg.loc[langs, COL_CPU_ENERGY],
        agg.loc[langs, COL_MEM_ENERGY],
        color=PARADIGM_COLORS[paradigm], label=paradigm, s=80, zorder=3
    )
    for lang in langs:
        ax.annotate(lang,
                    (agg.loc[lang, COL_CPU_ENERGY], agg.loc[lang, COL_MEM_ENERGY]),
                    textcoords='offset points', xytext=(6, 4), fontsize=8)

ax.set_title('CPU Energy vs Memory Energy — mean per language (J)', fontsize=12)
ax.set_xlabel('CPU Energy (J)')
ax.set_ylabel('Memory Energy (J)')
ax.legend(title='Paradigm')
plt.tight_layout()
ps.save_fig(fig, '02_cpu_vs_mem_scatter')
plt.show()

> **Takeaway:** CPU and memory energy move together across languages, but a few JIT/interpreted runtimes draw disproportionate DRAM energy for their CPU cost.

## 4. Paradigm Comparison

**Violin plots** show the full distribution shape per paradigm.
**Kruskal-Wallis** tests whether any paradigm differs significantly.
If significant, **pairwise Mann-Whitney U** tests with **Bonferroni correction** identify which pairs differ.
Effect size is reported as **rank-biserial correlation** r = 1 − 2U/(n₁·n₂).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, col, label in zip(axes,
                           [COL_CPU_ENERGY, COL_MEM_ENERGY],
                           ['CPU Energy (J)', 'Memory Energy (J)']):
    groups = [df[df['paradigm'] == p][col].values for p in PARADIGM_ORDER]
    parts  = ax.violinplot(groups, positions=range(len(PARADIGM_ORDER)), showmedians=True, showmeans=True)
    for pc, p in zip(parts['bodies'], PARADIGM_ORDER):
        pc.set_facecolor(PARADIGM_COLORS[p])
        pc.set_alpha(0.7)
    ax.set_xticks(range(len(PARADIGM_ORDER)))
    ax.set_xticklabels(PARADIGM_ORDER)
    ax.set_title(f'{label} by Paradigm')
    ax.set_ylabel(label)

fig.suptitle('Energy Distribution by Execution Paradigm', fontsize=13)
plt.tight_layout()
ps.save_fig(fig, '02_energy_violin_paradigm')
plt.show()

> **Takeaway:** the violins confirm the paradigm gap; Kruskal-Wallis (below) tests whether it is statistically significant.

In [ ]:
def rank_biserial(x, y):
    """Rank-biserial correlation as effect size for Mann-Whitney U."""
    u, _ = stats.mannwhitneyu(x, y, alternative='two-sided')
    return 1 - (2 * u) / (len(x) * len(y))

for col, label in [(COL_CPU_ENERGY, 'CPU Energy (J)'), (COL_MEM_ENERGY, 'Memory Energy (J)')]:
    groups   = {p: df[df['paradigm'] == p][col].values for p in PARADIGM_ORDER}
    kw_stat, kw_p = stats.kruskal(*groups.values())
    n_pairs = len(PARADIGM_ORDER) * (len(PARADIGM_ORDER) - 1) // 2

    print(f"\n{'='*60}")
    print(f"{label}")
    print(f"  Kruskal-Wallis H={kw_stat:.3f}, p={kw_p:.4f} ", end='')
    print("(SIGNIFICANT)" if kw_p < ALPHA else "(not significant)")

    if kw_p < ALPHA:
        print(f"  Post-hoc Mann-Whitney U (Bonferroni α={ALPHA/n_pairs:.4f}):")
        for (p1, p2) in combinations(PARADIGM_ORDER, 2):
            u, p = stats.mannwhitneyu(groups[p1], groups[p2], alternative='two-sided')
            p_adj = min(p * n_pairs, 1.0)
            r = rank_biserial(groups[p1], groups[p2])
            sig = "✓" if p_adj < ALPHA else "✗"
            print(f"    {sig} {p1} vs {p2}: U={u:.0f}, p_adj={p_adj:.4f}, r={r:.3f}")

## 5. Per-Benchmark Energy Heatmap

Heatmap of mean CPU and Memory energy (J) for each language × benchmark combination
(the per-cell means stored in `results_clean_runs.csv`). This reveals which benchmarks are
most energy-intensive and which languages suffer disproportionately on specific workloads.

In [ ]:
# Per-cell means come directly from df_mean (results_clean.csv).
pivot_cpu = df_mean.pivot(index='language', columns='benchmark', values=COL_CPU_ENERGY)
pivot_mem = df_mean.pivot(index='language', columns='benchmark', values=COL_MEM_ENERGY)

lang_sort = lang_means(COL_CPU_ENERGY).sort_values().index
pivot_cpu = pivot_cpu.loc[lang_sort]
pivot_mem = pivot_mem.loc[lang_sort]

fig, axes = plt.subplots(1, 2, figsize=(18, 9))
for ax, pivot, title in zip(axes,
                              [pivot_cpu, pivot_mem],
                              ['CPU Energy (J) — mean', 'Memory Energy (J) — mean']):
    sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax,
                linewidths=0.3, cbar_kws={'label': 'Energy (J)'})
    ax.set_title(title, fontsize=11)
    ax.set_xlabel('Benchmark')
    ax.set_ylabel('Language')
    plt.setp(ax.get_xticklabels(), rotation=45, ha='right')

fig.suptitle('Per-Benchmark Energy Heatmap — sorted by mean CPU energy (J)', fontsize=13)
plt.tight_layout()
ps.save_fig(fig, '02_energy_heatmap_benchmark')
plt.show()

> **Takeaway:** k-nucleotide and regex-redux are the most energy-intensive benchmarks, and the interpreted languages suffer most on them.

## 6. Energy Efficiency Ranking

Languages ranked by **mean** CPU energy (J) ascending (two-step mean, equal benchmark
weight) — lower rank = more efficient. A combined rank averages CPU and Memory ranks.

In [ ]:
rank_agg = lang_means([COL_CPU_ENERGY, COL_MEM_ENERGY]).copy()
rank_agg.columns = ['cpu_mean_J', 'mem_mean_J']
rank_agg.insert(0, 'paradigm', rank_agg.index.map(PARADIGM))
rank_agg['cpu_rank'] = rank_agg['cpu_mean_J'].rank().astype(int)
rank_agg['mem_rank'] = rank_agg['mem_mean_J'].rank().astype(int)
rank_agg['combined_rank'] = ((rank_agg['cpu_rank'] + rank_agg['mem_rank']) / 2).round(1)
ranking = rank_agg.sort_values('combined_rank')
ranking.index.name = 'Language'
ranking[['paradigm', 'cpu_mean_J', 'mem_mean_J', 'cpu_rank', 'mem_rank', 'combined_rank']]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ranked = ranking.sort_values('cpu_mean_J')   # order bars by the plotted metric
colors = [PARADIGM_COLORS[PARADIGM[l]] for l in ranked.index]
bars = ax.barh(ranked.index, ranked['cpu_mean_J'], color=colors, alpha=0.85, edgecolor='white')

# value label at the end of each bar
x_max = ranked['cpu_mean_J'].max()
for bar, val in zip(bars, ranked['cpu_mean_J']):
    ax.text(bar.get_width() + x_max * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:,.0f}', va='center', ha='left', fontsize=8, color='#333333')
ax.set_xlim(0, x_max * 1.12)

ax.set_title('CPU Energy Efficiency Ranking — mean across all benchmarks (J)', fontsize=12)
ax.set_xlabel('Mean CPU Energy (J)')
ax.set_ylabel('Language')
ax.invert_yaxis()   # most efficient (lowest) at the top
legend_handles = [mpatches.Patch(color=PARADIGM_COLORS[p], label=p, alpha=0.85)
                  for p in PARADIGM_ORDER]
ax.legend(handles=legend_handles, title='Paradigm', loc='upper right')
plt.tight_layout()
ps.save_fig(fig, '02_cpu_energy_ranking')
plt.show()

> **Takeaway:** the final CPU-energy ranking is led by the AOT native compilers and trailed by the interpreted languages.

### Memory Energy Ranking

Same view for **mean memory (DRAM) energy (J)** — lower is more efficient. Ordered by the plotted metric, so the order differs from the CPU ranking.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ranked_mem = ranking.sort_values('mem_mean_J')   # order bars by the plotted metric
colors = [PARADIGM_COLORS[PARADIGM[l]] for l in ranked_mem.index]
bars = ax.barh(ranked_mem.index, ranked_mem['mem_mean_J'], color=colors, alpha=0.85, edgecolor='white')

# value label at the end of each bar
x_max = ranked_mem['mem_mean_J'].max()
for bar, val in zip(bars, ranked_mem['mem_mean_J']):
    ax.text(bar.get_width() + x_max * 0.01, bar.get_y() + bar.get_height() / 2,
            f'{val:,.2f}', va='center', ha='left', fontsize=8, color='#333333')
ax.set_xlim(0, x_max * 1.12)

ax.set_title('Memory Energy Efficiency Ranking — mean across all benchmarks (J)', fontsize=12)
ax.set_xlabel('Mean Memory Energy (J)')
ax.set_ylabel('Language')
ax.invert_yaxis()   # most efficient (lowest) at the top
legend_handles = [mpatches.Patch(color=PARADIGM_COLORS[p], label=p, alpha=0.85)
                  for p in PARADIGM_ORDER]
ax.legend(handles=legend_handles, title='Paradigm', loc='upper right')
plt.tight_layout()
ps.save_fig(fig, '02_mem_energy_ranking')
plt.show()

> **Takeaway:** memory energy is dominated by Erlang (driven by its very long regex-redux run); the rest cluster low, so the DRAM ranking differs markedly from the CPU-energy ranking.